# Track D / Day 3 — Name-collision filter (Colab)

Implements `pilot_0_1_execution_spec.md` §2.2 step 3: *"String/fuzzy match against all 200 TOFU author names AND against a real-author list (no real people). Drop collisions."*

Two reference sets, per the frozen decisions in `ghosts/DECISIONS.md`:
1. All 200 TOFU author names — reuses Day 1's own validated name-recovery function directly (imports `day1_schema_extraction.py`, doesn't reimplement it).
2. Real authors from Wikidata (`DECISIONS.md` item 2) — humans with occupation writer or any subclass. Queried via **QLever** (a faster mirror of the same Wikidata data) rather than `query.wikidata.org` directly, because the ~600K-row pull is not tractable on Wikidata's own endpoint within reasonable time — documented in the script's own module docstring, not a change to what's queried.

Matching rule: `rapidfuzz.fuzz.token_sort_ratio >= 85` (case-folded, punctuation-stripped) OR an exact surname match — either drops the WHOLE ghost author (all 20 rows), not just one row.

**No paid API calls in this script at all** — Wikidata/QLever and the TOFU dataset are both free, public data. The only real cost here is time (the Wikidata pull can take a while — see step 5).

## 1. Mount Drive and clone/pull the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/unlearning_pilot'
REPO_DIR = os.path.join(PROJECT_DIR, 'unlearning-audit-study')
os.makedirs(PROJECT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/shravanidhus31/unlearning-audit-study.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log -1 --oneline

## 2. Confirm the Day 2 data and Day 3 script are present

In [ ]:
import os
for p in ['scripts/day3_collision_filter.py', 'scripts/day1_schema_extraction.py',
          'ghosts/candidates_raw.jsonl']:
    assert os.path.exists(p), f'{p} not found -- re-run cell 1, or check the repo state.'
print('All required files present.')

## 3. Install dependencies

In [ ]:
!pip install -q rapidfuzz datasets

## 4. Self-test (offline — no network, sanity-check before the real run)

In [ ]:
!python scripts/day3_collision_filter.py --selftest

## 5. Full run

This is the real Day 3 run. It:
- Loads all 4,000 TOFU rows and recovers all 200 author names (fast, seconds).
- Pulls the Wikidata real-author reference list via QLever — **this is the slow part**, roughly 300-600+ paginated requests across the ~340 writer-subclass categories. Expect this to take a while (tens of minutes); it's cached afterward to `ghosts/wikidata_authors_cache.json` so you only pay this cost once — re-runs reuse the cache automatically unless you pass `--refresh-wikidata`.
- Fuzzy-matches all 30 ghost authors against both reference sets, drops any collision (whole author, all 20 rows), and checks the 30 ghosts don't collide with each other either.

If you want a **much faster first look** before committing to the full Wikidata pull, run once with `--skip-wikidata` first (TOFU-only check, seconds) — but that is a *partial* result, not the final Day 3 output; the real run below still needs to complete before Day 4.

In [ ]:
# Optional quick partial check first (seconds, TOFU names only):
!python scripts/day3_collision_filter.py --skip-wikidata --outdir ghosts

In [ ]:
# The real run -- this is the one that matters. Can take a while (see markdown above).
!python scripts/day3_collision_filter.py --outdir ghosts

## 6. Read the results

In [ ]:
with open('ghosts/collision_report.md', encoding='utf-8') as f:
    report = f.read()
# Print the summary + collision-check sections rather than the whole (potentially long) report.
start = report.find('## 4. Collision check')
print(report[start:])

## 7. Commit the results manually
This notebook does not push. Review `ghosts/collision_report.md` and `ghosts/candidates_filtered.jsonl` yourself first, then run the printed commands (in a terminal, or uncomment the cell below).

In [ ]:
import json
rows = [json.loads(l) for l in open('ghosts/candidates_filtered.jsonl', encoding='utf-8')]
authors = {r['author_id'] for r in rows}
print(f'{len(rows)} surviving QA rows across {len(authors)} surviving authors (need >= 400 rows for Day 5\'s trim)')

print()
print('To commit manually:')
print('  git add ghosts/collision_report.md ghosts/candidates_filtered.jsonl ghosts/wikidata_authors_cache.json')
print('  git commit -m "Track D Day 3: name-collision filter (TOFU 200 + Wikidata real authors)"')
print('  git push')